In [1]:
# Mount Drive, clone repos, patch, load both models
from google.colab import drive
drive.mount('/content/drive')

import os, sys, torch, numpy as np
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU — switch to T4 GPU runtime before continuing.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# UPDATED PATHS TO 70_10_20
STGCN_CKPT      = '/content/drive/MyDrive/HRC_Research/checkpoints/stgcn_hri30_70_10_20/best_stgcn_hri30.pt'
CTRGCN_CKPT     = '/content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30_70_10_20/best_ctrgcn_hri30.pt'
STGCN_TEST_DATA = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format/test_data.npy'
STGCN_TEST_LABEL= '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format/test_label.pkl'
CTRGCN_TEST_NPZ = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/ctrgcn_format/HRI30_CS.npz'
OUTPUT_DIR      = '/content/drive/MyDrive/HRC_Research/results/latency'
os.makedirs(OUTPUT_DIR, exist_ok=True)

checks = [
    ("ST-GCN checkpoint",  STGCN_CKPT),
    ("CTR-GCN checkpoint", CTRGCN_CKPT),
    ("ST-GCN test data",   STGCN_TEST_DATA),
    ("ST-GCN test labels", STGCN_TEST_LABEL),
    ("CTR-GCN npz",        CTRGCN_TEST_NPZ),
]
all_ok = True
for name, path in checks:
    if os.path.exists(path):
        print(f"  OK      {name}")
    else:
        print(f"  MISSING {name}: {path}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Fix missing files before continuing.")


STGCN_DIR = '/content/st-gcn'
if not os.path.exists(STGCN_DIR):
    os.system(f'git clone https://github.com/yysijie/st-gcn.git {STGCN_DIR}')
stgcn_io = Path(f'{STGCN_DIR}/torchlight/torchlight/io.py')
text = stgcn_io.read_text()
if 'weights_only=False' not in text:
    stgcn_io.write_text(text.replace(
        'torch.load(weights_path)',
        'torch.load(weights_path, weights_only=False, map_location="cpu")'
    ))
os.system(f'pip install -e {STGCN_DIR}/torchlight -q')

sys.path.insert(0, STGCN_DIR)
from net.st_gcn import Model as STGCN_Model

stgcn_model = STGCN_Model(
    in_channels=3, num_class=30, dropout=0.5,
    edge_importance_weighting=True,
    graph_args={'layout': 'ntu-rgb+d', 'strategy': 'spatial'}
)
ckpt = torch.load(STGCN_CKPT, map_location='cpu', weights_only=False)
state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
state = {k[7:] if k.startswith('module.') else k: v for k, v in state.items()}
stgcn_model.load_state_dict(state, strict=True)
stgcn_model = stgcn_model.to(device).eval()
print("ST-GCN loaded ✓")


CTRGCN_DIR = '/content/CTR-GCN'
if not os.path.exists(CTRGCN_DIR):
    os.system(f'git clone https://github.com/Uason-Chen/CTR-GCN.git {CTRGCN_DIR}')
os.system('pip install -q tensorboardX torchpack fvcore iopath yacs thop')
os.system(f'pip install -e {CTRGCN_DIR}/torchlight -q')

ctrgcn_util = Path(f'{CTRGCN_DIR}/torchlight/torchlight/util.py')
text = ctrgcn_util.read_text()
if 'PaviLogger = None' not in text:
    ctrgcn_util.write_text(text.replace(
        'from torchpack.runner.hooks import PaviLogger',
        'try:\n    from torchpack.runner.hooks import PaviLogger\nexcept ImportError:\n    PaviLogger = None'
    ))

sys.path.insert(0, CTRGCN_DIR)
from model.ctrgcn import Model as CTRGCN_Model

ctrgcn_model = CTRGCN_Model(
    num_class=30, num_point=25, num_person=1,
    graph='graph.ntu_rgb_d.Graph',
    graph_args={'labeling_mode': 'spatial'}
)
ckpt = torch.load(CTRGCN_CKPT, map_location='cpu', weights_only=False)
state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
state = {k[7:] if k.startswith('module.') else k: v for k, v in state.items()}
ctrgcn_model.load_state_dict(state, strict=True)
ctrgcn_model = ctrgcn_model.to(device).eval()
print("CTR-GCN loaded ✓")

print(f"\nBoth models ready on {device}.")

Mounted at /content/drive
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
  OK      ST-GCN checkpoint
  OK      CTR-GCN checkpoint
  OK      ST-GCN test data
  OK      ST-GCN test labels
  OK      CTR-GCN npz
ST-GCN loaded ✓
CTR-GCN loaded ✓

Both models ready on cuda.


In [2]:
# Load test data
import numpy as np, pickle

stgcn_test_data = np.load(STGCN_TEST_DATA)
with open(STGCN_TEST_LABEL, 'rb') as f:
    _, y_test = pickle.load(f)
stgcn_test_labels = np.array(y_test)

npz = np.load(CTRGCN_TEST_NPZ)
ctrgcn_test_data   = npz['x_test']
ctrgcn_test_labels = npz['y_test']

assert stgcn_test_data.shape[0] == ctrgcn_test_data.shape[0]
assert (stgcn_test_labels == ctrgcn_test_labels).all()

N = stgcn_test_labels.shape[0]
print(f"ST-GCN  test data shape: {stgcn_test_data.shape}")
print(f"CTR-GCN test data shape: {ctrgcn_test_data.shape}")
print(f"\n{N} test samples loaded.")

ST-GCN  test data shape: (588, 3, 150, 25, 1)
CTR-GCN test data shape: (588, 3, 150, 25, 1)

588 test samples loaded.


In [3]:
# Latency Measurement
import torch
import numpy as np
import csv, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = '/content/drive/MyDrive/HRC_Research/results/latency'

if device.type != 'cuda':
    raise RuntimeError(
        "GPU not detected. Switch to T4 GPU runtime (Runtime > Change runtime type)."
    )

# Helper: softmax + entropy + consensus (CPU numpy, negligible time)
def softmax_np(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def entropy_np(probs):
    return -np.sum(probs * np.log(probs + 1e-9), axis=1)

def consensus_math(logits_st, logits_ctr):
    p_st  = softmax_np(logits_st)
    p_ctr = softmax_np(logits_ctr)
    H_st  = entropy_np(p_st);   H_ctr = entropy_np(p_ctr)
    w_st  = np.exp(-H_st);      w_ctr = np.exp(-H_ctr)
    s     = w_st + w_ctr
    w_st /= s;                  w_ctr /= s
    return np.argmax(w_st[:, None]*p_st + w_ctr[:, None]*p_ctr, axis=1)

# GPU timing function
def measure_gpu_latency_ms(fn, n_warmup=10, n_runs=50):
    """
    Runs fn() n_warmup times (discarded), then n_runs times timed.
    Returns mean and std latency in milliseconds.
    fn must be a callable with no arguments.
    """
    # Warmup — ensures GPU kernels are compiled and cached
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    return float(np.mean(times)), float(np.std(times))

# Prepare single-sample tensors (shape: 1, 3, 150, 25, 1)
single_st  = torch.tensor(stgcn_test_data[:1],   dtype=torch.float32).to(device)
single_ctr = torch.tensor(ctrgcn_test_data[:1],  dtype=torch.float32).to(device)

print("=" * 60)
print("LATENCY MEASUREMENT — Single Sample on T4 GPU")
print("Protocol: 10 warmup + 50 timed runs")
print("=" * 60)

# (A) ST-GCN alone
print("\n[A] Measuring ST-GCN alone...")
with torch.no_grad():
    st_mean_ms, st_std_ms = measure_gpu_latency_ms(
        lambda: stgcn_model(single_st)
    )
print(f"    ST-GCN:  {st_mean_ms:.2f} ± {st_std_ms:.2f} ms per sample")
print(f"             {1000/st_mean_ms:.1f} samples/sec")

# (B) CTR-GCN alone
print("\n[B] Measuring CTR-GCN alone...")
with torch.no_grad():
    ctr_mean_ms, ctr_std_ms = measure_gpu_latency_ms(
        lambda: ctrgcn_model(single_ctr)
    )
print(f"    CTR-GCN: {ctr_mean_ms:.2f} ± {ctr_std_ms:.2f} ms per sample")
print(f"             {1000/ctr_mean_ms:.1f} samples/sec")

# (C) Full pipeline: both models + consensus math
print("\n[C] Measuring full pipeline (ST-GCN + CTR-GCN + consensus)...")

def full_pipeline():
    with torch.no_grad():
        logits_st  = stgcn_model(single_st).cpu().numpy()
        logits_ctr = ctrgcn_model(single_ctr).cpu().numpy()
    consensus_math(logits_st, logits_ctr)

pipe_mean_ms, pipe_std_ms = measure_gpu_latency_ms(full_pipeline)
print(f"    Pipeline:{pipe_mean_ms:.2f} ± {pipe_std_ms:.2f} ms per sample")
print(f"             {1000/pipe_mean_ms:.1f} samples/sec")

# Throughput on full test set
print("\n[D] Full test set throughput (batch_size=64, 588 samples)...")

import time

def run_full_testset_st():
    with torch.no_grad():
        for i in range(0, stgcn_test_data.shape[0], 64):
            b = torch.tensor(stgcn_test_data[i:i+64], dtype=torch.float32).to(device)
            stgcn_model(b)

def run_full_testset_ctr():
    with torch.no_grad():
        for i in range(0, ctrgcn_test_data.shape[0], 64):
            b = torch.tensor(ctrgcn_test_data[i:i+64], dtype=torch.float32).to(device)
            ctrgcn_model(b)

torch.cuda.synchronize()
t0 = time.time()
run_full_testset_st()
torch.cuda.synchronize()
st_throughput_sec = time.time() - t0

torch.cuda.synchronize()
t0 = time.time()
run_full_testset_ctr()
torch.cuda.synchronize()
ctr_throughput_sec = time.time() - t0

N = stgcn_test_data.shape[0]
print(f"    ST-GCN  full test set: {st_throughput_sec:.2f}s  "
      f"({N/st_throughput_sec:.1f} samples/sec)")
print(f"    CTR-GCN full test set: {ctr_throughput_sec:.2f}s  "
      f"({N/ctr_throughput_sec:.1f} samples/sec)")

# Summary
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  ST-GCN  alone:    {st_mean_ms:.2f} ms/sample  "
      f"({1000/st_mean_ms:.1f} Hz)")
print(f"  CTR-GCN alone:    {ctr_mean_ms:.2f} ms/sample  "
      f"({1000/ctr_mean_ms:.1f} Hz)")
print(f"  Full pipeline:    {pipe_mean_ms:.2f} ms/sample  "
      f"({1000/pipe_mean_ms:.1f} Hz)")
print(f"  Typical HRC camera: 30 fps = 33.3 ms/frame")
real_time = "YES" if pipe_mean_ms < 33.3 else "NO"
print(f"  Pipeline < 33.3ms? {real_time} → Real-time capable: {real_time}")
print("=" * 60)

# Save to Drive
# UPDATED FILENAME WITH SUFFIX
CSV_PATH = os.path.join(OUTPUT_DIR, 'latency_results_70_10_20.csv')
with open(CSV_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'configuration', 'mean_ms', 'std_ms',
        'samples_per_sec', 'real_time_at_30fps'
    ])
    writer.writerow([
        'ST-GCN alone',
        round(st_mean_ms, 3), round(st_std_ms, 3),
        round(1000/st_mean_ms, 1), st_mean_ms < 33.3
    ])
    writer.writerow([
        'CTR-GCN alone',
        round(ctr_mean_ms, 3), round(ctr_std_ms, 3),
        round(1000/ctr_mean_ms, 1), ctr_mean_ms < 33.3
    ])
    writer.writerow([
        'Full pipeline (ST-GCN + CTR-GCN + consensus)',
        round(pipe_mean_ms, 3), round(pipe_std_ms, 3),
        round(1000/pipe_mean_ms, 1), pipe_mean_ms < 33.3
    ])

print(f"\nCSV saved to Drive: {CSV_PATH}")

LATENCY MEASUREMENT — Single Sample on T4 GPU
Protocol: 10 warmup + 50 timed runs

[A] Measuring ST-GCN alone...
    ST-GCN:  11.33 ± 3.92 ms per sample
             88.3 samples/sec

[B] Measuring CTR-GCN alone...
    CTR-GCN: 80.27 ± 45.96 ms per sample
             12.5 samples/sec

[C] Measuring full pipeline (ST-GCN + CTR-GCN + consensus)...
    Pipeline:70.04 ± 30.15 ms per sample
             14.3 samples/sec

[D] Full test set throughput (batch_size=64, 588 samples)...
    ST-GCN  full test set: 1.87s  (314.3 samples/sec)
    CTR-GCN full test set: 2.60s  (226.0 samples/sec)

SUMMARY
  ST-GCN  alone:    11.33 ms/sample  (88.3 Hz)
  CTR-GCN alone:    80.27 ms/sample  (12.5 Hz)
  Full pipeline:    70.04 ms/sample  (14.3 Hz)
  Typical HRC camera: 30 fps = 33.3 ms/frame
  Pipeline < 33.3ms? NO → Real-time capable: NO

CSV saved to Drive: /content/drive/MyDrive/HRC_Research/results/latency/latency_results_70_10_20.csv
